[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/proyecto/guia_proyecto.ipynb) · [Ver en el sitio](https://hesusg.github.io/dammg-supervisado/notebooks/proyecto/guia_proyecto.html) · [Menú de notebooks](https://hesusg.github.io/dammg-supervisado/notebooks/)

# Guía del proyecto · MiBici de punta a punta
**DAMMG · Módulo VI · Aprendizaje supervisado**

Este notebook recorre, en orden, todo lo que necesita el proyecto final: plantear el problema, partir los datos sin hacer trampa, construir un baseline, entrenar y comparar modelos (lineales, logística, árboles, ensambles y SVM), elegir la métrica y el umbral, evaluar en test una sola vez, leer los resultados con la teoría y nombrar los riesgos.

**Cómo está armado.** Cada sección empieza con un **ejemplo resuelto** que ya corre con datos reales de MiBici. Después vienen **sus opciones**: decisiones que ustedes toman (qué problema, qué variables, qué modelos, qué umbral) cambiando una o dos líneas. Al final de cada sección hay un recuadro **Para su reporte** con lo que tienen que escribir.

**Si eligen un problema y llegan al final, tienen el proyecto casi completo.** Lo que falta es lo que ninguna plantilla puede hacer por ustedes: justificar sus decisiones con sus propios números.

| Sección | Tema | Criterio de la rúbrica |
|---|---|---|
| 1 | El problema y los datos | C1 |
| 2 | Partición honesta, fugas y baseline | C2 |
| 3 | Lo que cada modelo supone (sesgo inductivo) | C5 |
| 4 | Regresión lineal: costo, gradiente y Ridge | C3 |
| 5 | Clasificación: logística, matriz de confusión y AP | C3, C4 |
| 6 | Árboles, bosques y boosting | C3, C5 |
| 7 | Máquinas de soporte vectorial | C3, C5 |
| 8 | La comparativa: ¿quién gana y la diferencia es real? | C3, C5 |
| 9 | Métrica y umbral para operar | C4 |
| 10 | Test, una sola vez | C4 |
| 11 | *Opcional:* modelos en GPU | (extra) |
| 12 | Limitaciones y riesgos | C6 |
| 13 | Del notebook al reporte | todos |

**Tiempo de ejecución.** Con el problema B y el entorno gratuito de Colab (CPU), el notebook completo corre en unos 5 minutos. Con los problemas C o D (panel completo, 4 millones de filas) tarda entre 10 y 15.

## 0 · Preparación

> **Antes de escribir nada**
> 1. *Archivo*, opción *Guardar una copia en Drive* y trabaja en esa copia; si no, al cerrar la pestaña pierdes todo.
> 2. Ejecuta las celdas **en orden**. Si reinicias el entorno o algo falla raro: *Entorno de ejecución*, opción *Reiniciar y ejecutar todo*.
> 3. Escribe dentro de las celdas que ya existen (no copies/pegues celdas): así no pierden la etiqueta que usa el evaluador.
>
> *Palabras en inglés que usamos tal cual:* **baseline** (línea base: el modelo tonto que hay que vencer) · **pipeline** (cadena de preprocesamiento + modelo) · **proxy** (variable sustituta de lo que de verdad queremos medir) · **bootstrap** (remuestreo con reemplazo) · **chat waterfall** (todos escriben su respuesta en el chat y la envían al mismo tiempo) · **precision** y **recall** (los nombres de sklearn para precisión y exhaustividad).

In [ ]:
# L0 · Preparación del entorno. No necesitas modificar esta celda.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'b54216735cb8f868fae885c42b527a232fd11154cfd352dc9881ac1831b3a5aa',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 6.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES
import time
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, average_precision_score,
                             precision_score, recall_score, confusion_matrix, accuracy_score,
                             log_loss, precision_recall_curve)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

---
# 1 · El problema y los datos

## 1.1 · Ejemplo: la operación de MiBici

MiBici es el sistema de bicicletas públicas del área metropolitana de Guadalajara. Cada estación tiene un número fijo de anclajes (su **capacidad**). Cuando una estación se queda **vacía**, quien llega no encuentra bici; cuando se queda **llena**, quien llega no puede devolverla. Para evitarlo, el sistema mueve bicis en camioneta de las estaciones que sobran a las que faltan (**rebalanceo**).

Las camionetas y el personal son limitados. Un modelo útil le dice a la operación **dónde** y **cuándo** mandar a la cuadrilla, o **cuántas** bicis va a pedir cada estación mañana. Esa decisión es la que el proyecto tiene que ayudar a tomar.

Los datos son agregados por **estación y hora**: cada fila dice cuántas bicis salieron (`departures`) y llegaron (`arrivals`) en una estación durante una hora, más variables construidas con el pasado de esa estación.

In [ ]:
df_curso = pd.read_parquet(RUTA_DATOS)
print(f"{len(df_curso):,} filas · {df_curso['station_id'].nunique()} estaciones · "
      f"{df_curso['hour'].min():%d-%m-%Y} a {df_curso['hour'].max():%d-%m-%Y}")
df_curso.head()

Antes de modelar, miren los datos. A la izquierda, una semana de la estación con más movimiento; a la derecha, el promedio de salidas por hora del día en las 40 estaciones.

In [ ]:
est = df_curso.groupby("station_id")["departures"].sum().idxmax()
semana = df_curso[(df_curso["station_id"] == est) & (df_curso["hour"] >= "2026-06-08") & (df_curso["hour"] < "2026-06-15")]
perfil = df_curso.groupby(["hour_of_day", "is_weekend"])["departures"].mean().unstack()

fig, ax = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1.6, 1]})
ax[0].plot(semana["hour"], semana["departures"], label="salidas")
ax[0].plot(semana["hour"], semana["arrivals"], label="llegadas", alpha=0.8)
ax[0].set(title=f"Estación {est}: semana del lunes 8 de junio", ylabel="bicis por hora")
ax[0].legend()
ax[1].plot(perfil.index, perfil[0], marker="o", label="entre semana")
ax[1].plot(perfil.index, perfil[1], marker="o", label="fin de semana")
ax[1].set(title="Salidas promedio por hora", xlabel="hora del día", ylabel="bicis por hora")
ax[1].legend()
plt.tight_layout()
plt.show()

Dos cosas saltan a la vista: hay **picos de entrada y salida al trabajo** entre semana (7–9 h y 17–20 h), y el fin de semana tiene otro patrón. Cualquier modelo que no sepa la hora del día va a tener problemas.

## 1.2 · La regla de oro: ¿a qué hora conocerían cada variable?

En producción, el modelo predice **antes** de que pase lo que quiere predecir. Por eso, para cada columna pregúntense: *¿a qué hora exacta tendría este dato si el modelo estuviera corriendo de verdad?*

| Columna | Qué es | ¿Se puede usar como entrada? |
|---|---|---|
| `departures`, `arrivals`, `net_flow` | salidas, llegadas y salidas − llegadas **de la hora t** | **No.** Son de la misma hora que queremos predecir |
| `future_net` | salidas − llegadas en las **próximas 3 h**, $[t, t+3)$ | **No.** Es el futuro: de aquí salen las etiquetas |
| `departures_lag_24h`, `departures_lag_168h` | salidas a la misma hora hace 1 día y hace 1 semana | Sí |
| `departures_mean_24h`, `departures_mean_168h` | promedio de salidas de las últimas 24 h y 168 h, hasta hace un día | Sí |
| `net_lag_24h`, `net_lag_168h`, `net_mean_24h` | lo mismo para el flujo neto | Sí |
| `hour_of_day`, `day_of_week`, `is_weekend`, `month` | calendario | Sí: se sabe con anticipación |
| `capacity`, `station_id` | características fijas de la estación | Sí |

## 1.3 · Sus opciones: elijan el problema

Los cuatro problemas usan los mismos datos de MiBici y el resto del notebook se adapta solo. Elijan uno y escriban su letra en la celda de abajo.

| | Problema | Tipo | Datos | Decisión que apoya |
|---|---|---|---|---|
| **A** | **Pronóstico de demanda.** ¿Cuántas bicis van a salir de cada estación a cada hora, con un día de anticipación? | regresión (MAE o RMSE) | recorte: 40 estaciones, jun–sep 2026 | cuántas bicis tener en cada estación mañana |
| **B** | **Alerta de desbalance.** ¿Esta estación va a tener un desbalance fuerte, \|`future_net`\| ≥ 6, en las próximas 3 h? (≈ 17 % de las horas) | clasificación | recorte | a qué estaciones mandar la camioneta este turno |
| **C** | **Riesgo de vaciado o llenado.** ¿El desbalance de las próximas 3 h va a ser de al menos la mitad de la capacidad de la estación? (`risk`, ≈ 2 %) | clasificación de evento raro | panel completo: 469 estaciones, oct 2025–sep 2026 | lo mismo, en toda la red |
| **D** | **Riesgo de quedarse sin bicis.** Igual que C, pero solo cuando salen más de las que llegan (`risk_empty`, ≈ 1 %) | clasificación de evento muy raro | panel completo | a dónde **llevar** bicis antes de que falten |
| **E** | **Problema propio** con otros datos (sin datos personales) | el que aplique | los suyos | la suya |

**¿Cuál elegir?** B es la más directa y la más rápida de correr. A es la opción para quien prefiera regresión. C y D son las más realistas (toda la red, eventos raros) y las más exigentes: con 2 % o 1 % de positivos, la elección de métrica y umbral pesa mucho más. Si eligen E, lean la nota al final de esta sección.

In [ ]:
PROBLEMA = "B"     # ✋ EQUIPO: "A", "B", "C" o "D"

PROBLEMAS = {
    "A": {"nombre": "Pronóstico de salidas por estación-hora, con 24 h de anticipación", "tipo": "regresion", "datos": "recorte"},
    "B": {"nombre": "Alerta de desbalance |future_net| ≥ umbral en las próximas 3 h", "tipo": "clasificacion", "datos": "recorte"},
    "C": {"nombre": "Riesgo de vaciado o llenado (risk) en toda la red", "tipo": "clasificacion", "datos": "panel"},
    "D": {"nombre": "Riesgo de quedarse sin bicis (risk_empty) en toda la red", "tipo": "clasificacion", "datos": "panel"},
}
cfg = PROBLEMAS[PROBLEMA]
ES_REGRESION = cfg["tipo"] == "regresion"
EN_PANEL = cfg["datos"] == "panel"

if EN_PANEL:
    df = pd.read_parquet(_obtener("mibici_panel_completo.parquet", "37c815f15cd5b8b9713c36705fa8b58746060a5211bd3683cc1befada94cea79"))   # unos 40 MB
else:
    df = df_curso
N_ESTACIONES = df["station_id"].nunique()
print(f"Problema {PROBLEMA}: {cfg['nombre']}")
print(f"{len(df):,} filas · {N_ESTACIONES} estaciones · {df['hour'].min():%d-%m-%Y} a {df['hour'].max():%d-%m-%Y}")

> **Si eligen E (problema propio).** Carguen sus datos en la celda de arriba en lugar de MiBici y adapten tres funciones de la sección 2: `objetivo` (la variable a predecir), `variables` (las entradas, todas conocibles antes de predecir) y la partición. Todo lo demás (modelos, comparativa, umbral, test) funciona igual mientras `y` sea numérica (regresión) o 0/1 (clasificación). Necesitan una columna de tiempo para partir por fechas, o una justificación de por qué una partición aleatoria es honesta en su caso.

> **Para su reporte (C1).**
> - **Pregunta:** qué decisión se toma con la predicción, quién la toma y cada cuánto.
> - **Variable objetivo y unidad de predicción:** por ejemplo, "una fila = una estación en una hora; la etiqueta es 1 si en las 3 horas siguientes el desbalance llega a 6 bicis".
> - **Límites de los datos:** son viajes agregados por hora; no se observa cuántas bicis había en la estación (el inventario), ni el clima, ni los rebalanceos que ya hizo la operación, ni la demanda de quien llegó y no encontró bici.

---
# 2 · Partición honesta, fugas y baseline

## 2.1 · Por qué no se parte al azar

Con datos en el tiempo, una partición aleatoria pone en entrenamiento la hora de las 8:00 y en validación la de las 9:00 del mismo día y la misma estación. El modelo "ve" el día que tiene que predecir y el número de validación sale optimista. En producción no hay esa suerte: siempre se predice un futuro que no se ha visto.

Con las variables de este notebook, que ya miran por lo menos 24 h atrás, una partición al azar infla poco el número. Pero basta **una** variable mal calculada para inflarlo mucho, como van a ver en la sección 2.4.

En todo el notebook usamos la partición **temporal** que ya trae la columna `particion`:

| Datos | train | val | test |
|---|---|---|---|
| recorte (problemas A y B) | jun–jul 2026 | ago 2026 | sep 2026 |
| panel completo (C y D) | oct 2025–jun 2026 | jul 2026 | ago–sep 2026 |

**Reglas del juego:**
1. Todo lo que se *aprende* de los datos (el umbral del evento, el escalador, una imputación) se calcula **solo con train**.
2. Los hiperparámetros, el modelo y el umbral se eligen con **validación**.
3. **Test se usa una vez**, al final (sección 10).

## 2.2 · La variable objetivo y la partición de su problema

In [ ]:
tr_completo = df[df["particion"] == "train"]
va_completo = df[df["particion"] == "val"]
te_completo = df[df["particion"] == "test"]

if PROBLEMA == "B":
    # El umbral del evento se aprende de los datos: percentil 85 de |future_net| SOLO en train
    UMBRAL_Y = float(tr_completo["future_net"].abs().quantile(0.85))
    print(f"umbral del evento (percentil 85 de train) = {UMBRAL_Y}")


def objetivo(d):
    if PROBLEMA == "A":
        return d["departures"].astype(float)
    if PROBLEMA == "B":
        return (d["future_net"].abs() >= UMBRAL_Y).astype(int)
    if PROBLEMA == "C":
        return d["risk"].astype(int)
    if PROBLEMA == "D":
        return d["risk_empty"].astype(int)


for nombre, d in [("train", tr_completo), ("val", va_completo), ("test", te_completo)]:
    y = objetivo(d)
    resumen_y = f"media = {y.mean():.2f} bicis/h" if ES_REGRESION else f"prevalencia = {y.mean():.2%}"
    print(f"{nombre:5s} · {len(d):>9,} filas · {d['hour'].min():%d-%m-%Y} a {d['hour'].max():%d-%m-%Y} · {resumen_y}")

## 2.3 · Sus opciones: qué variables usar

Las variables de partida miran por lo menos 24 h atrás, así que todas se conocen a tiempo. Ustedes eligen cuánto calendario agregar:

| `CONJUNTO` | Qué agrega | Cuándo conviene |
|---|---|---|
| `"basico"` | solo rezagos y promedios del pasado | para ver cuánto aporta el calendario |
| `"calendario"` | + hora del día, día de la semana, fin de semana, hora pico | **recomendado**: los árboles aprovechan bien `hour_of_day` |
| `"calendario+horas"` | + una columna 0/1 por cada hora del día | si el modelo lineal o la logística son importantes para ustedes: una sola pendiente sobre `hour_of_day` no puede dibujar dos picos, 24 indicadoras sí |

**Opción `USAR_HISTORICO`: convertir la experiencia en una variable.** Los rezagos dicen qué pasó ayer o hace una semana, pero ningún modelo sabe *qué estación es* ni cómo se comporta normalmente a esa hora. La variable `historico` es la tasa del evento (o la demanda media) de **esta estación, a esta hora, entre semana o en fin de semana**, calculada con train. En los datos de train se calcula **sin la semana de la propia fila**, para que su etiqueta no se cuele en su variable. Es la misma información que usa el baseline "típico de estación-hora" de la sección 2.5. Prueben con `False` y con `True`: sin ella, es común que ningún modelo le gane a ese baseline.

Para los problemas C y D también entran rezagos que solo trae el panel completo (`departures_lag_336h`, `net_mean_168h`, `future_net_lag_168h`, `risk_lag_168h`). Pueden agregar las suyas en `variables`: para cada una, repitan la pregunta de la regla de oro.

In [ ]:
CONJUNTO = "calendario"          # ✋ EQUIPO: "basico", "calendario" o "calendario+horas"
HORAS_PICO = [7, 8, 9, 17, 18, 19, 20]
SUBMUESTRA_TRAIN = 300_000       # ✋ solo aplica al panel completo (3 millones de filas en train)


def variables(d):
    if ES_REGRESION:
        X = d[["departures_lag_24h", "departures_lag_168h", "departures_mean_24h", "departures_mean_168h"]].copy()
    else:
        X = pd.DataFrame({
            "net_lag_24h": d["net_lag_24h"], "net_lag_168h": d["net_lag_168h"], "net_mean_24h": d["net_mean_24h"],
            "abs_net_lag_24h": d["net_lag_24h"].abs(), "abs_net_lag_168h": d["net_lag_168h"].abs(),
            "abs_net_mean_24h": d["net_mean_24h"].abs(),
            "departures_mean_24h": d["departures_mean_24h"], "departures_mean_168h": d["departures_mean_168h"],
            "departures_lag_168h": d["departures_lag_168h"], "capacity": d["capacity"],
        }, index=d.index)
    if EN_PANEL:
        X["departures_lag_336h"] = d["departures_lag_336h"]
        X["net_mean_168h"] = d["net_mean_168h"]
        X["future_net_lag_168h"] = d["future_net_lag_168h"]                    # [t−168, t−165): ya pasó
        X["future_net_lag_168h_rel"] = d["future_net_lag_168h"] / d["capacity"]
        X["risk_lag_168h"] = d["risk_lag_168h"]
    if CONJUNTO in ("calendario", "calendario+horas"):
        X["hour_of_day"] = d["hour_of_day"]
        X["day_of_week"] = d["day_of_week"]
        X["is_weekend"] = d["is_weekend"]
        X["pico"] = d["hour_of_day"].isin(HORAS_PICO).astype(int)
    if CONJUNTO == "calendario+horas":
        for h in range(24):
            X[f"h_{h:02d}"] = (d["hour_of_day"] == h).astype(int)
    return X.astype(float)


def prepara(d):
    # Regresa (filas, X, y) sin huecos. En el panel, las primeras horas de cada estación no tienen rezagos.
    X, y = variables(d), objetivo(d)
    ok = X.notna().all(axis=1) & y.notna() & d["future_net"].notna()
    return d[ok], X[ok], y[ok].to_numpy()


tr_base = tr_completo
if EN_PANEL and len(tr_completo) > SUBMUESTRA_TRAIN:
    tr_base = tr_completo.sample(SUBMUESTRA_TRAIN, random_state=0).sort_values(["hour", "station_id"])
tr, X_tr, y_tr = prepara(tr_base)
va, X_va, y_va = prepara(va_completo)
te, X_te, y_te = prepara(te_completo)

USAR_HISTORICO = True            # ✋ EQUIPO: True o False (ver arriba)
CLAVE_HIST = ["station_id", "hour_of_day", "is_weekend"]
if USAR_HISTORICO:
    inicio = tr_completo["hour"].min()
    ref = tr_completo.assign(_y=objetivo(tr_completo).to_numpy(), _sem=(tr_completo["hour"] - inicio).dt.days // 7)
    ref = ref[ref["_y"].notna()]
    total = ref.groupby(CLAVE_HIST)["_y"].agg(["sum", "count"])            # SOLO train
    por_semana = ref.groupby(CLAVE_HIST + ["_sem"])["_y"].agg(["sum", "count"])
    media_global = float(ref["_y"].mean())
    tasa_train = (total["sum"] / total["count"]).rename("_h")

    def historico(d, es_train):
        if es_train:   # filas de train: sin la semana de la propia fila
            j = (d[CLAVE_HIST].assign(_sem=(d["hour"] - inicio).dt.days // 7)
                 .join(total, on=CLAVE_HIST).join(por_semana, on=CLAVE_HIST + ["_sem"], rsuffix="_sem"))
            resto = (j["count"] - j["count_sem"].fillna(0)).replace(0, np.nan)
            return ((j["sum"] - j["sum_sem"].fillna(0)) / resto).fillna(media_global).to_numpy()
        return d[CLAVE_HIST].join(tasa_train, on=CLAVE_HIST)["_h"].fillna(media_global).to_numpy()   # val y test

    X_tr["historico"] = historico(tr, True)
    X_va["historico"] = historico(va, False)
    X_te["historico"] = historico(te, False)


def prepara_train_completo():
    # Todo train, sin submuestra, con las mismas variables (para las opciones de GPU)
    filas, X, y = prepara(tr_completo)
    if USAR_HISTORICO:
        X["historico"] = historico(filas, True)
    return filas, X, y

escalador = StandardScaler().fit(X_tr)           # se ajusta SOLO con train
Z_tr, Z_va, Z_te = (escalador.transform(X) for X in (X_tr, X_va, X_te))
print(f"{X_tr.shape[1]} variables:", ", ".join(X_tr.columns))
print(f"train {len(X_tr):,} · val {len(X_va):,} · test {len(X_te):,} filas")

## 2.4 · Ejemplo: así se ve una fuga

Una forma barata de cazar fugas: medir qué tan bien predice **cada variable sola**. Si una sola variable le gana a todo lo demás, desconfíen. Para mostrarlo, metemos a propósito dos fugas:

- `net_flow`: el flujo de la hora *t*, que en producción todavía no se conoce.
- Una **tasa histórica mal calculada**: la de cada estación, hora y día de la semana, pero usando **todos los meses**, incluidos validación y test. Es un error muy común: la variable parece "experiencia", pero ya contiene las respuestas de agosto.

In [ ]:
def calidad_sola(x, y):
    # Qué tan bien ordena una variable sola (en cualquiera de los dos sentidos)
    if ES_REGRESION:
        return abs(np.corrcoef(x, y)[0, 1])
    return max(average_precision_score(y, x), average_precision_score(y, -x))


auditoria = {col: calidad_sola(X_va[col].to_numpy(), y_va) for col in X_va.columns}
auditoria["net_flow  ⚠ (hora t: FUGA)"] = calidad_sola(va["net_flow"].abs().to_numpy() if not ES_REGRESION else va["net_flow"].to_numpy(), y_va)
if ES_REGRESION:
    auditoria["arrivals  ⚠ (hora t: FUGA)"] = calidad_sola(va["arrivals"].to_numpy(), y_va)
CLAVE_FUGA = ["station_id", "hour_of_day", "day_of_week"]
todos = pd.concat([tr_completo, va_completo, te_completo])
tasa_todos = todos.assign(_y=objetivo(todos).to_numpy()).groupby(CLAVE_FUGA)["_y"].mean().rename("_f")
auditoria["tasa con TODOS los meses  ⚠ (FUGA)"] = calidad_sola(va.join(tasa_todos, on=CLAVE_FUGA)["_f"].to_numpy(), y_va)
del todos
nombre_col = "|correlación| en val" if ES_REGRESION else "AP en val, variable sola"
pd.Series(auditoria, name=nombre_col).sort_values(ascending=False).round(3).to_frame()

Las fugas quedan arriba. Un modelo con ellas se vería espectacular en validación y sería inútil en producción (o, en el caso de la tasa, se vería mejor de lo que es).

Fíjense también en `historico`: está cerca de las fugas y **no** es una. Se calcula solo con train, que en producción ya pasó. La tabla no decide sola; sirve para señalar sospechosas. Para cada una, repitan la pregunta de la regla de oro: *¿a qué hora exacta conocería este dato?* Si al agregar una variable suya el número mejora "demasiado", vuelvan aquí.

## 2.5 · Ejemplo: los baselines

Un **baseline** es una regla sin aprendizaje que el modelo tiene que vencer. Si no le gana, el modelo no sirve, por elegante que sea. Usamos dos:

- **El que no hace nada:** la media de train (regresión) o "nunca hay alerta" (clasificación; su AP es la prevalencia).
- **La costumbre:** lo que pasó a esta hora la semana pasada, o el valor típico de **esta estación a esta hora** en train. Es lo que haría alguien con experiencia en la operación y una hoja de cálculo.

In [ ]:
if ES_REGRESION:
    METRICA_REG = "MAE"      # ✋ EQUIPO: "MAE" o "RMSE" (ver la sección 4.4 antes de decidir)
    NOMBRE_METRICA = METRICA_REG
    MAYOR_ES_MEJOR = False

    def metrica(y, s):
        return mean_absolute_error(y, s) if METRICA_REG == "MAE" else float(np.sqrt(mean_squared_error(y, s)))
else:
    NOMBRE_METRICA = "AP"
    MAYOR_ES_MEJOR = True
    metrica = average_precision_score


def es_mejor(a, b):
    return a > b if MAYOR_ES_MEJOR else a < b


CLAVE = ["station_id", "hour_of_day", "is_weekend"]
_ref = tr_completo.assign(_y=objetivo(tr_completo).to_numpy())          # todo train (sin submuestra)
tipico = _ref.groupby(CLAVE)["_y"]
tipico = tipico.median() if ES_REGRESION else tipico.mean()     # MAE premia la mediana; para eventos, la tasa
relleno = float(_ref["_y"].median()) if ES_REGRESION else float(_ref["_y"].mean())

BASELINES = {
    "nada (media de train)" if ES_REGRESION else "nada (nunca alerta)":
        lambda d: np.full(len(d), y_tr.mean()) if ES_REGRESION else np.zeros(len(d)),
    "semana pasada": (lambda d: d["departures_lag_168h"].to_numpy()) if ES_REGRESION else
                     (lambda d: d["net_lag_168h"].abs().to_numpy()) if PROBLEMA == "B" else
                     (lambda d: (d["future_net_lag_168h"].abs() / d["capacity"]).to_numpy()) if PROBLEMA == "C" else
                     (lambda d: (d["future_net_lag_168h"] / d["capacity"]).to_numpy()),
    "típico de estación-hora": lambda d: d.join(tipico.rename("_t"), on=CLAVE)["_t"].fillna(relleno).to_numpy(),
}

registro = []        # una fila por cada (familia, hiperparámetros) probados
elegidos = {}        # el mejor de cada familia en validación
I_EVAL_TR = np.random.default_rng(1).choice(len(X_tr), min(50_000, len(X_tr)), replace=False)   # train para medir


def _guarda(fila, info):
    global registro
    registro = [r for r in registro if (r["familia"], r["hiperparámetros"]) != (fila["familia"], fila["hiperparámetros"])]
    registro.append(fila)
    f = fila["familia"]
    if f not in elegidos or es_mejor(fila[f"{NOMBRE_METRICA} val"], elegidos[f]["val"]):
        elegidos[f] = {"val": fila[f"{NOMBRE_METRICA} val"], "hiper": fila["hiperparámetros"], **info}


for nombre, regla in BASELINES.items():
    _guarda({"familia": "baseline · " + nombre, "hiperparámetros": "-",
             f"{NOMBRE_METRICA} train": metrica(y_tr, regla(tr)), f"{NOMBRE_METRICA} val": metrica(y_va, regla(va)),
             "segundos": 0.0},
            {"modelo": None, "regla": regla, "escalado": False, "s_va": regla(va)})
pd.DataFrame(registro).set_index("familia").round(3)

El baseline "típico de estación-hora" es difícil de vencer, y eso es una buena noticia: es un rival honesto. Si su mejor modelo le gana por poco, la sección 8 les dice si la diferencia es real.

> **Para su reporte (C2).** La partición (fechas de train, val y test) y por qué es temporal; qué se ajustó solo con train; cómo revisaron fugas (la tabla de 2.4); el baseline que hay que vencer **con su número**.

## 2.6 · La herramienta para comparar modelos

De aquí en adelante, cada modelo se entrena con `evalua(familia, hiperparámetros, modelo, escalado)`. La función ajusta en train, mide en train y en validación con la métrica de su problema, guarda la fila en `registro` y se queda con el mejor de cada familia en `elegidos`. Así la rejilla completa queda en el notebook, que es lo que pide la rúbrica, no solo el ganador.

In [ ]:
def puntaje(modelo, A):
    # Regresión: la predicción. Clasificación: un puntaje que ordena (probabilidad o distancia a la frontera)
    if ES_REGRESION:
        return modelo.predict(A)
    if hasattr(modelo, "predict_proba"):
        return modelo.predict_proba(A)[:, 1]
    return modelo.decision_function(A)


def _filas(A, idx):
    return A.iloc[idx] if hasattr(A, "iloc") else A[idx]


def evalua(familia, hiper, modelo, escalado, filas=None, A=None, y=None, **kw_fit):
    # filas: entrenar solo con una muestra de train (SVM). A, y: otros datos de entrenamiento (GPU).
    A_tr, A_va = (Z_tr, Z_va) if escalado else (X_tr, X_va)
    A_fit, y_fit = (A, y) if A is not None else (A_tr, y_tr)
    if filas is not None:
        A_fit, y_fit = _filas(A_fit, filas), y_fit[filas]
    t0 = time.perf_counter()
    modelo.fit(A_fit, y_fit, **kw_fit)
    seg = time.perf_counter() - t0
    s_va = puntaje(modelo, A_va)
    fila = {"familia": familia, "hiperparámetros": hiper,
            f"{NOMBRE_METRICA} train": metrica(y_tr[I_EVAL_TR], puntaje(modelo, _filas(A_tr, I_EVAL_TR))),
            f"{NOMBRE_METRICA} val": metrica(y_va, s_va), "segundos": round(seg, 1)}
    _guarda(fila, {"modelo": modelo, "regla": None, "escalado": escalado, "s_va": s_va})
    print(f"{familia:24s} {hiper:30s} {NOMBRE_METRICA} train {fila[NOMBRE_METRICA + ' train']:.3f} · "
          f"val {fila[NOMBRE_METRICA + ' val']:.3f} · {seg:5.1f} s")
    return modelo


def muestra_train(n, semilla=0):
    # Índices de una muestra de train; en clasificación, estratificada para conservar la prevalencia
    rng = np.random.default_rng(semilla)
    n = min(n, len(y_tr))
    if ES_REGRESION:
        return np.sort(rng.choice(len(y_tr), n, replace=False))
    pos, neg = np.flatnonzero(y_tr == 1), np.flatnonzero(y_tr == 0)
    k = max(1, round(n * len(pos) / len(y_tr)))
    return np.sort(np.concatenate([rng.choice(pos, k, replace=False), rng.choice(neg, n - k, replace=False)]))


print("listo: evalua(), muestra_train()")

---
# 3 · Lo que cada modelo supone (sesgo inductivo)

## 3.1 · Ejemplo: tres modelos, los mismos datos, tres futuros distintos

Ningún modelo aprende "solo de los datos". Cada uno trae una **apuesta** sobre cómo se ve el mundo fuera de los ejemplos que vio: eso es su **sesgo inductivo**. Se nota sobre todo donde no hay datos.

Un experimento con todas las estaciones y una sola variable: el promedio de salidas de la semana anterior. Entrenamos tres modelos **solo con las estaciones-hora tranquilas** (promedio de 6 salidas o menos) y les pedimos que predigan las movidas, que nunca vieron. Como sí tenemos esos datos, podemos comparar contra lo que de verdad pasó (puntos negros).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

junio_julio = df_curso[df_curso["particion"] == "train"]
CORTE = 6
x1, y1 = junio_julio[["departures_mean_168h"]], junio_julio["departures"]
visto = (x1["departures_mean_168h"] <= CORTE).to_numpy()
rejilla_x = pd.DataFrame({"departures_mean_168h": np.linspace(0, x1.values.max(), 300)})
no_visto = junio_julio[~visto]
real = no_visto.groupby(no_visto["departures_mean_168h"].round())["departures"].mean()

modelos_sesgo = {
    "lineal: el mundo es una recta": LinearRegression(),
    "árbol: el mundo es una escalera": DecisionTreeRegressor(max_depth=3, min_samples_leaf=200),
    "vecinos: se parece a lo cercano": KNeighborsRegressor(n_neighbors=200),
}
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(x1[visto], y1[visto], s=4, alpha=0.08, color=COLORES["w"], edgecolors="none")
for nombre, m in modelos_sesgo.items():
    m.fit(x1[visto], y1[visto])
    ax.plot(rejilla_x, m.predict(rejilla_x), label=nombre)
ax.scatter(real.index, real.values, color=COLORES["tinta"], zorder=5, label="lo que pasó (promedio)")
ax.axvspan(CORTE, x1.values.max(), color=COLORES["amarillo"], alpha=0.15, label="no lo vieron al entrenar")
ax.set(ylim=(0, 25), xlabel="promedio de salidas de la semana anterior", ylabel="salidas en la hora",
       title="Tres apuestas fuera de lo observado")
ax.legend(fontsize=9, loc="upper left")
plt.show()
print(f"{len(no_visto):,} estaciones-hora ({len(no_visto) / len(junio_julio):.0%}) quedaron fuera del entrenamiento")

Dentro de lo observado los tres modelos dicen casi lo mismo. Fuera, se separan: la recta sigue subiendo, el árbol y los vecinos se quedan planos en el último valor que conocen. **Aquí** la apuesta lineal acertó, porque la demanda de verdad crece con su promedio. Con otra variable (la hora del día, que tiene dos picos) la recta sería la que falla. Ningún modelo tiene razón por definición: depende de cómo sea el mundo, y esa es la idea que tienen que usar para explicar sus resultados.

| Modelo | Qué supone (su apuesta) | Le ayuda cuando… | Le estorba cuando… |
|---|---|---|---|
| Regresión lineal / Ridge | la respuesta cambia en línea recta con cada variable; Ridge además prefiere pesos chicos | hay relación aditiva y pocos datos | hay interacciones (la hora importa distinto entre semana) |
| Regresión logística | la **probabilidad** (en escala log-odds) es lineal en las variables: la frontera es un plano | la señal es monótona: más desbalance ayer, más riesgo hoy | el riesgo depende de combinaciones (hora pico **y** estación chica) |
| Árbol de decisión | el espacio se parte en cajas con preguntas sí/no; dentro de cada caja todo es igual | hay umbrales e interacciones | crece demasiado y memoriza: varianza alta |
| Bosque aleatorio | promediar muchos árboles distintos cancela su ruido | el árbol solo sobreajusta | la señal es muy suave o lineal |
| Boosting | cada árbol corrige lo que los anteriores fallaron | hay mucha señal y suficiente validación para detenerlo | se deja correr sin control: memoriza |
| SVM | importa la frontera con **margen** amplio, definida por pocos puntos (vectores de soporte); con kernel la frontera se curva | fronteras suaves en datos escalados | millones de filas: el costo crece como $m^2$ o más |

> **Para su reporte (C5, primera mitad).** Una frase por cada modelo que usen: qué supone y por qué esa apuesta le ayudó o le estorbó **en su problema**. Lo van a poder confirmar con la tabla de la sección 8.

---
# 4 · Regresión lineal: costo, gradiente y Ridge

## 4.1 · Ejemplo: el costo y el descenso de gradiente, a mano

Con una variable, la regresión lineal es $\hat y = w\,x + b$ y el costo es el error cuadrático medio

$$J(w,b) = \frac{1}{2m}\sum_{i=1}^{m}\big(w\,x^{(i)} + b - y^{(i)}\big)^2.$$

El descenso de gradiente da pasos de tamaño $\alpha$ en la dirección que más baja $J$. Lo corremos con NumPy sobre las salidas del recorte (todas las estaciones, train) y una variable: el promedio de salidas de la semana anterior. Primero **escalamos** $x$ (media 0, desviación 1): así un mismo $\alpha$ sirve para $w$ y para $b$.

In [ ]:
ej = df_curso[df_curso["particion"] == "train"]
x_raw = ej["departures_mean_168h"].to_numpy(float)
y_ej = ej["departures"].to_numpy(float)
mu, sigma = x_raw.mean(), x_raw.std()
x_esc = (x_raw - mu) / sigma

fig, ax = plt.subplots(figsize=(8, 4))
for alpha in [0.05, 1.0, 2.05]:
    camino = du.descenso_gradiente(x_esc, y_ej, w0=0, b0=0, alpha=alpha, iters=60)
    J = [paso[2] for paso in camino]
    ax.plot(J, marker=".", label=f"α = {alpha}")
    print(f"α = {alpha:<5} → J final = {J[-1]:.4g}")
ax.set(yscale="log", xlabel="iteración", ylabel="J(w, b)", title="Tres tasas de aprendizaje")
ax.legend()
plt.show()

w_gd, b_gd, _ = du.descenso_gradiente(x_esc, y_ej, 0, 0, alpha=1.0, iters=60)[-1]
w_orig, b_orig = w_gd / sigma, b_gd - w_gd * mu / sigma                # regresamos a la escala original
w_ols, b_ols = du.ols_1d(x_raw, y_ej)
print(f"descenso de gradiente: w = {w_orig:.4f}, b = {b_orig:.4f}")
print(f"solución cerrada:      w = {w_ols:.4f}, b = {b_ols:.4f}")

Con $\alpha$ chica converge pero lento; con $\alpha = 1$ llega en pocos pasos; con $\alpha$ arriba de 2 (con $x$ escalada) cada paso se pasa del mínimo y $J$ explota. El resultado coincide con la solución cerrada. `sklearn` resuelve esto mismo por dentro, con más variables y sin que tengamos que elegir $\alpha$.

## 4.2 · Ejemplo: Ridge y el equilibrio entre sesgo y varianza

Ridge agrega una penalización a los pesos grandes: $J(w,b) + \frac{\lambda}{2m}\lVert w\rVert^2$ (en `sklearn` el parámetro se llama `alpha` y equivale a nuestra $\lambda$). Ayuda cuando hay **muchas variables para pocos datos**. Para verlo, entrenamos con solo 400 horas y muchas variables indicadoras (una por estación, por hora y por día).

In [ ]:
from sklearn.linear_model import Ridge

def indicadoras(d):
    partes = [d[["departures_lag_24h", "departures_lag_168h", "departures_mean_24h", "departures_mean_168h"]]]
    for col, valores in [("station_id", sorted(df_curso["station_id"].unique())), ("hour_of_day", range(24)), ("day_of_week", range(7))]:
        partes.append(pd.DataFrame({f"{col}_{v}": (d[col] == v).astype(float) for v in valores}, index=d.index))
    return pd.concat(partes, axis=1)

poco = ej.sample(400, random_state=3)
ej_va = df_curso[df_curso["particion"] == "val"]
A_poco, A_val = indicadoras(poco), indicadoras(ej_va)
print(f"{A_poco.shape[1]} variables para {len(poco)} filas")

lambdas = np.logspace(-2, 3.5, 12)
curva = []
for lam in lambdas:
    r = Ridge(alpha=lam).fit(A_poco, poco["departures"])
    curva.append((mean_absolute_error(poco["departures"], r.predict(A_poco)),
                  mean_absolute_error(ej_va["departures"], r.predict(A_val))))
curva = np.array(curva)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lambdas, curva[:, 0], marker="o", label="train", color=COLORES["w"])
ax.plot(lambdas, curva[:, 1], marker="o", label="validación", color=COLORES["error"])
ax.axvline(lambdas[curva[:, 1].argmin()], color=COLORES["lambda"], ls="--", label="mejor λ en validación")
ax.set(xscale="log", xlabel="λ", ylabel="MAE (bicis/h)", title="Ridge con 400 filas: de varianza alta a sesgo alto")
ax.legend()
plt.show()

A la izquierda ($\lambda$ chica) el error de train es bajo y el de validación alto: **varianza alta**, el modelo memoriza las 400 filas. A la derecha ($\lambda$ grande) los dos errores suben juntos: **sesgo alto**, el modelo quedó demasiado rígido. El mejor $\lambda$ está en medio y **lo elige validación**.

Con todo el train (decenas de miles de filas y pocas variables) Ridge casi no cambia nada, porque casi no hay varianza que controlar. No es un error: es el mismo diagnóstico leído al revés.

## 4.3 · Sus opciones: el modelo lineal de su problema

Si su problema es **A**, aquí entrenan su primer modelo: Ridge con una rejilla de $\lambda$. Si es B, C o D, su modelo lineal es la regresión logística de la sección 5 y esta celda no hace nada.

In [ ]:
if ES_REGRESION:
    for lam in [0.1, 10, 1_000, 30_000]:                 # ✋ EQUIPO: pueden cambiar la rejilla
        evalua("lineal (Ridge)", f"λ={lam:g}", Ridge(alpha=lam), escalado=True)
else:
    print("Su problema es de clasificación: el modelo lineal es la logística (sección 5).")

## 4.4 · Ejemplo: ¿MAE o RMSE?

Solo aplica a regresión, pero vale la pena leerlo. El **MAE** cobra cada bici de error igual; el **RMSE** eleva al cuadrado y castiga mucho más los errores grandes. ¿Qué tan grandes son los errores grandes aquí?

In [ ]:
err = ej_va["departures"] - ej_va["departures_lag_168h"]       # errores del baseline "semana pasada" en agosto
print(f"MAE  = {err.abs().mean():.2f} bicis/h")
print(f"RMSE = {np.sqrt((err ** 2).mean()):.2f} bicis/h")
print(f"El 1 % de las horas con más error acumula el {err.abs().nlargest(len(err) // 100).sum() / err.abs().sum():.0%} del error absoluto total")

El RMSE sale bastante más alto que el MAE: hay pocas horas con errores muy grandes (eventos, días festivos, una estación que cerró) y el RMSE las pesa mucho más. La pregunta para el equipo es de **costo**, no de estadística:

- Si a la operación le cuesta lo mismo cada bici que falta, sin importar si son 2 o 20: **MAE**.
- Si quedarse muy corto en una hora pico es desproporcionadamente caro (decenas de personas sin bici, quejas): **RMSE**.

Decídanlo **antes** de comparar modelos y cámbienlo en `METRICA_REG` (sección 2.5). Con MAE, la mediana es la predicción que minimiza el error; con RMSE, la media.

> **Para su reporte (C3, C4).** Para el problema A: el $\lambda$ elegido y su número en validación; MAE o RMSE y el argumento de costo que lo justifica.

---
# 5 · Clasificación: logística, matriz de confusión y AP

## 5.1 · Ejemplo: la trampa del accuracy

Volvemos a la alerta de desbalance del recorte (el problema B), con cuatro variables. La regresión logística entrega una **probabilidad**: $\hat p = \sigma(w\cdot x + b)$, con $\sigma(z) = 1/(1+e^{-z})$. Se entrena minimizando la **log-loss** (entropía cruzada), que es lo mismo que maximizar la verosimilitud de una Bernoulli.

Primero, el número que no hay que reportar: el accuracy.

In [ ]:
from sklearn.linear_model import LogisticRegression

def cuatro(d):
    X = pd.DataFrame({"abs_net_mean_24h": d["net_mean_24h"].abs(), "abs_net_lag_168h": d["net_lag_168h"].abs(),
                      "departures_mean_168h": d["departures_mean_168h"], "pico": d["hour_of_day"].isin(HORAS_PICO).astype(int)})
    return X, (d["future_net"].abs() >= UMBRAL_EVENTO).astype(int).to_numpy()

Xe_tr, ye_tr = cuatro(ej)
Xe_va, ye_va = cuatro(ej_va)
esc_e = StandardScaler().fit(Xe_tr)
log_e = LogisticRegression(max_iter=1000).fit(esc_e.transform(Xe_tr), ye_tr)
p_va = log_e.predict_proba(esc_e.transform(Xe_va))[:, 1]

print(f"prevalencia del evento en val:         {ye_va.mean():.1%}")
print(f"accuracy de 'nunca hay alerta':        {accuracy_score(ye_va, np.zeros_like(ye_va)):.1%}")
print(f"accuracy de la logística (umbral 0.5): {accuracy_score(ye_va, p_va >= 0.5):.1%}")
print(f"log-loss · predecir siempre la prevalencia de train: {log_loss(ye_va, np.full(len(ye_va), ye_tr.mean())):.3f}")
print(f"log-loss · logística:                                {log_loss(ye_va, p_va):.3f}")

"Nunca hay alerta" tiene un accuracy altísimo y no sirve para nada: no manda la camioneta a ningún lado. Con clases desbalanceadas, el accuracy premia ignorar a la clase que nos importa. La log-loss sí muestra que la logística aprendió algo.

## 5.2 · Ejemplo: la matriz de confusión, precision y recall

In [ ]:
a = p_va >= 0.5
mc = confusion_matrix(ye_va, a)
print(pd.DataFrame(mc, index=["real: sin evento", "real: evento"], columns=["predice: no", "predice: alerta"]))
print(f"\nprecision = {precision_score(ye_va, a):.2f}  →  de cada 100 alertas, cuántas eran evento de verdad")
print(f"recall    = {recall_score(ye_va, a):.2f}  →  de cada 100 eventos, cuántos atrapamos")

- **Falso positivo** (alerta sin evento): la camioneta va y no hacía falta. Cuesta gasolina y una hora de cuadrilla.
- **Falso negativo** (evento sin alerta): la estación se vacía o se llena y nadie fue. Cuesta usuarios sin servicio.

El umbral de 0.5 no tiene nada de especial. Moverlo cambia cuántas alertas se mandan y, con eso, la precision y el recall.

## 5.3 · Ejemplo: la curva precision–recall y el AP

La **curva precision–recall** muestra todos los umbrales a la vez. Su área es la **AP** (*average precision*): resume qué tan bien ordena el modelo, sin comprometerse con un umbral. Un modelo al azar tiene AP igual a la prevalencia. Por eso la usamos para **elegir modelo**, y el umbral se fija aparte, con el costo o el presupuesto (sección 9).

In [ ]:
prec, rec, umb = precision_recall_curve(ye_va, p_va)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(rec, prec, color=COLORES["w"], label=f"logística · AP = {average_precision_score(ye_va, p_va):.3f}")
ax[0].axhline(ye_va.mean(), color=COLORES["tinta"], ls="--", label=f"al azar · AP = {ye_va.mean():.3f}")
ax[0].set(xlabel="recall", ylabel="precision", title="Curva precision–recall (val)")
ax[0].legend()
alertas = np.array([(p_va >= t).mean() for t in umb])
ax[1].plot(umb, prec[:-1], label="precision", color=COLORES["w"])
ax[1].plot(umb, rec[:-1], label="recall", color=COLORES["error"])
ax[1].plot(umb, alertas, label="% de horas con alerta", color=COLORES["lambda"], ls="--")
ax[1].set(xlabel="umbral sobre p̂", title="Lo que cambia al mover el umbral")
ax[1].legend()
plt.tight_layout()
plt.show()

## 5.4 · Sus opciones: la logística de su problema

Si su problema es de clasificación (B, C o D), aquí entra su primer modelo. `C` es el inverso de la regularización ($C = 1/\lambda$): $C$ chica, pesos chicos.

**Opción `BALANCEAR`.** Con eventos raros (C y D) la logística tiende a dar probabilidades muy bajas a todo. `class_weight="balanced"` pesa más a los positivos al entrenar. Cambia las probabilidades (ya no son calibradas) pero a veces mejora el orden, que es lo que mide la AP. Pruébenlo y quédense con lo que diga validación.

In [ ]:
BALANCEAR = False     # ✋ EQUIPO: True para probar class_weight="balanced"

if not ES_REGRESION:
    peso = "balanced" if BALANCEAR else None
    for C in [0.01, 0.1, 1, 10]:                        # ✋ EQUIPO: pueden cambiar la rejilla
        evalua("logística", f"C={C}" + (" bal" if BALANCEAR else ""),
               LogisticRegression(C=C, max_iter=2000, class_weight=peso), escalado=True)
else:
    print("Su problema es de regresión: el modelo lineal es Ridge (sección 4.3).")

Si todas las $C$ dan casi lo mismo, no es un error: con decenas de miles de filas y pocas variables la regularización casi no tiene varianza que controlar (lo mismo que pasó con Ridge en 4.2).

> **Para su reporte (C3, C4).** Por qué no reportan accuracy; la AP de la logística contra la AP del baseline; el $C$ elegido.

---
# 6 · Árboles, bosques y boosting

## 6.1 · Ejemplo: la mejor pregunta, calculada a mano

Un árbol hace preguntas sí/no. En cada nodo elige la pregunta que deja los grupos **más puros**. La **entropía** de un nodo con proporción $p$ de positivos es $H = -p\log_2 p - (1-p)\log_2(1-p)$; la **ganancia de información** de un corte es la entropía del padre menos el promedio ponderado de la de los hijos. El índice de **Gini**, $1 - p^2 - (1-p)^2$, mide lo mismo de otra forma.

Comparemos tres preguntas candidatas para la raíz, con los conteos de la alerta de desbalance en train.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

candidatas = {
    "¿es hora pico?": Xe_tr["pico"] == 1,
    "¿es fin de semana?": ej["is_weekend"].to_numpy() == 1,
    "¿|net_mean_24h| ≥ 1?": Xe_tr["abs_net_mean_24h"] >= 1,
}
padre = np.bincount(ye_tr, minlength=2)
filas = []
for pregunta, si in candidatas.items():
    si = np.asarray(si)
    izq, der = np.bincount(ye_tr[~si], minlength=2), np.bincount(ye_tr[si], minlength=2)
    filas.append({"pregunta": pregunta, "no (sin evento, evento)": tuple(izq), "sí (sin evento, evento)": tuple(der),
                  "ganancia (entropía)": du.ganancia_info(padre, izq, der, du.entropia),
                  "ganancia (Gini)": du.ganancia_info(padre, izq, der, du.gini)})
print(f"padre: {tuple(padre)} · entropía = {du.entropia(padre):.3f} · Gini = {du.gini(padre):.3f}")
pd.DataFrame(filas).set_index("pregunta").round(4)

De nuestras tres, gana la que más baja la entropía. El árbol de `sklearn` hace exactamente esta cuenta, pero para **todas** las variables y **todos** los cortes posibles. Si le pedimos un solo corte, ¿elige alguna de las nuestras?

In [ ]:
tocon = DecisionTreeClassifier(max_depth=1, criterion="entropy").fit(Xe_tr, ye_tr)
var_raiz, corte_raiz = Xe_tr.columns[tocon.tree_.feature[0]], tocon.tree_.threshold[0]
si = (Xe_tr[var_raiz] > corte_raiz).to_numpy()
g = du.ganancia_info(padre, np.bincount(ye_tr[~si], minlength=2), np.bincount(ye_tr[si], minlength=2), du.entropia)
print(f"sklearn elige: ¿{var_raiz} > {corte_raiz:.2f}? · ganancia (entropía) = {g:.4f}")

## 6.2 · Ejemplo: un árbol que crece hasta memorizar

Ahora sí con **su** problema. Dejamos crecer el árbol cada vez más y miramos train contra validación. Cada profundidad queda registrada en la comparativa como parte de la rejilla del árbol.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

Arbol = DecisionTreeRegressor if ES_REGRESION else DecisionTreeClassifier
profundidades = [2, 4, 6, 8, 12, 16, None]              # ✋ EQUIPO: pueden cambiar la rejilla
for p in profundidades:
    evalua("árbol", f"max_depth={p}", Arbol(max_depth=p, min_samples_leaf=5, random_state=0), escalado=False)

filas_arbol = [r for r in registro if r["familia"] == "árbol"]
fig, ax = plt.subplots(figsize=(8, 4))
etiquetas = [r["hiperparámetros"].split("=")[1] for r in filas_arbol]
ax.plot(etiquetas, [r[f"{NOMBRE_METRICA} train"] for r in filas_arbol], marker="o", label="train", color=COLORES["w"])
ax.plot(etiquetas, [r[f"{NOMBRE_METRICA} val"] for r in filas_arbol], marker="o", label="validación", color=COLORES["error"])
ax.set(xlabel="profundidad máxima", ylabel=NOMBRE_METRICA, title="Árbol: de sesgo alto a varianza alta")
ax.legend()
plt.show()

Con poca profundidad, train y validación están cerca pero son malos (**sesgo alto**). Con mucha, train sigue mejorando y validación se estanca o empeora (**varianza alta**): el árbol ya está memorizando ruido de junio y julio que no se repite en agosto.

## 6.3 · Ejemplo: muchos árboles, menos varianza

Dos formas de juntar árboles:

- **Bosque aleatorio:** muchos árboles profundos, cada uno con una muestra distinta de filas y de variables, que **votan**. Sus errores se cancelan en parte: baja la varianza.
- **Boosting:** árboles chicos en fila, cada uno entrenado para corregir lo que fallaron los anteriores. Baja el sesgo; la tasa de aprendizaje y el número de árboles controlan cuánto se acerca a memorizar.

## 6.4 · Sus opciones: bosque y boosting

Las rejillas son chicas a propósito (cada punto tarda de 5 a 40 s en Colab). Si quieren explorar más, cambien los valores y vuelvan a correr **solo esa celda**: la comparativa reemplaza las filas repetidas.

In [ ]:
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)

Bosque = RandomForestRegressor if ES_REGRESION else RandomForestClassifier
fraccion = min(1.0, 100_000 / len(X_tr))                # cada árbol ve a lo más 100 mil filas
for hoja in [5, 30]:                                    # ✋ EQUIPO: min_samples_leaf
    evalua("bosque aleatorio", f"hoja={hoja}, 150 árboles",
           Bosque(n_estimators=150, min_samples_leaf=hoja, max_features=0.5, max_samples=fraccion,
                  n_jobs=-1, random_state=0), escalado=False)

In [ ]:
Boosting = HistGradientBoostingRegressor if ES_REGRESION else HistGradientBoostingClassifier
perdida = {"loss": "absolute_error"} if ES_REGRESION and METRICA_REG == "MAE" else {}
for lr in [0.05, 0.15]:                                 # ✋ EQUIPO: tasa de aprendizaje
    for hojas in [15, 63]:                              # ✋ EQUIPO: hojas por árbol
        evalua("boosting", f"lr={lr}, hojas={hojas}",
               Boosting(learning_rate=lr, max_leaf_nodes=hojas, max_iter=300, early_stopping=False,
                        random_state=0, **perdida), escalado=False)

**Otras opciones que pueden probar** (una celda nueva por cada una, con el mismo patrón):
- Bosque con `max_depth` limitado en lugar de `min_samples_leaf`.
- Boosting con más árboles (`max_iter=600`) y tasa más baja: suele ganar un poco, tarda el doble.
- Árbol con `class_weight="balanced"` para C y D.

> **Para su reporte (C3, C5).** La gráfica de profundidad (o los números de train y val) como evidencia de sesgo y varianza; qué hiperparámetro movió a cada familia de un lado al otro.

---
# 7 · Máquinas de soporte vectorial

## 7.1 · Ejemplo: el margen y el papel de C

Una SVM busca la frontera con la **calle más ancha** entre las clases. Los puntos que tocan la orilla de la calle, o la invaden, son los **vectores de soporte**: solo ellos definen la frontera. Minimiza la pérdida **hinge**, $\max(0,\,1 - y\,f(x))$ con $y\in\{-1,+1\}$, más un término que premia el margen ancho. El parámetro $C$ dice cuánto cuesta invadir la calle:

- $C$ chica: calle ancha, se toleran errores, muchos vectores de soporte (más sesgo).
- $C$ grande: calle angosta, casi no se toleran errores (más varianza).

Para verlo en dos dimensiones usamos el **perfil** de cada estación a cada hora entre semana (40 estaciones × 24 horas = 960 puntos): cuántas bicis salen y llegan en promedio. Un punto es rojo si esa estación, a esa hora, tiene desbalance fuerte en al menos 30 % de los días. Las variables van en escala logarítmica y escaladas.

In [ ]:
from sklearn.svm import SVC, LinearSVC, SVR, LinearSVR

perfiles = (junio_julio[junio_julio["is_weekend"] == 0]
            .assign(evento=lambda d: (d["future_net"].abs() >= UMBRAL_EVENTO).astype(int))
            .groupby(["station_id", "hour_of_day"])
            .agg(salidas=("departures", "mean"), llegadas=("arrivals", "mean"), tasa=("evento", "mean")))
X2 = StandardScaler().fit_transform(np.log1p(perfiles[["salidas", "llegadas"]]))
y2 = (perfiles["tasa"] >= 0.3).astype(int).to_numpy()
print(f"{len(perfiles)} perfiles estación-hora · {y2.mean():.0%} con desbalance frecuente")


def dibuja_svm(m, ax, titulo):
    xx, yy = np.meshgrid(np.linspace(-2.5, 3.5, 250), np.linspace(-2.5, 3.5, 250))
    z = m.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, z > 0, levels=[-0.5, 0.5, 1.5], colors=[COLORES["datos"], "#fde0e0"], alpha=0.8)
    ax.contour(xx, yy, z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], linewidths=1.5)
    ax.scatter(X2[:, 0], X2[:, 1], c=np.where(y2 == 1, COLORES["error"], COLORES["w"]), s=14)
    ax.set(xlim=(-2.5, 3.5), ylim=(-2.5, 3.5), xlabel="salidas promedio (log, escalado)", ylabel="llegadas promedio (log, escalado)",
           title=f"{titulo}\n{int(m.n_support_.sum())} vectores de soporte · accuracy {m.score(X2, y2):.0%}")


fig, ax = plt.subplots(1, 3, figsize=(17, 5.5))
for k, (titulo, m) in enumerate([("lineal, C = 0.01", SVC(kernel="linear", C=0.01)),
                                  ("lineal, C = 10", SVC(kernel="linear", C=10)),
                                  ("kernel RBF, C = 10", SVC(kernel="rbf", C=10, gamma=1))]):
    dibuja_svm(m.fit(X2, y2), ax[k], titulo)
plt.tight_layout()
plt.show()

Los puntos se alinean sobre la diagonal (las estaciones donde salen muchas bicis también reciben muchas), y el desbalance frecuente aparece en las estaciones movidas y **lejos de la diagonal**. Una recta no puede encerrar eso:

- Con $C = 0.01$ la calle es tan ancha que el modelo prefiere no marcar casi nada como rojo.
- Con $C = 10$ la calle se angosta y la frontera se acomoda mejor, pero sigue siendo recta.
- Con **kernel RBF** la frontera se curva y rodea la zona azul: mismo algoritmo, otra apuesta.

Del accuracy de los títulos importa la diferencia entre modelos, no el número: aquí las clases están razonablemente balanceadas, pero con el evento por hora (sección 5) el accuracy engaña.

## 7.2 · Ejemplo: el truco del kernel, verificado

Un kernel calcula el producto punto en un espacio de más dimensiones **sin construir ese espacio**. Para el kernel polinomial de grado 2, $K(x,z) = (x\cdot z + 1)^2$ es igual a $\varphi(x)\cdot\varphi(z)$ con $\varphi(x) = (x_1^2,\ x_2^2,\ \sqrt2\,x_1x_2,\ \sqrt2\,x_1,\ \sqrt2\,x_2,\ 1)$. Comprobémoslo con dos perfiles reales:

In [ ]:
x, z = X2[0], X2[1]
print(f"x = {x.round(3)}, z = {z.round(3)}")
print(f"φ(x)·φ(z)   = {du.phi_poli2(x) @ du.phi_poli2(z):.6f}   (6 dimensiones, construidas a mano)")
print(f"(x·z + 1)²  = {(x @ z + 1) ** 2:.6f}   (2 dimensiones, sin construir nada)")

El RBF hace lo mismo con un espacio de dimensión infinita. El precio: la SVM con kernel necesita comparar cada fila con muchas otras, y su costo de entrenamiento crece entre $m^2$ y $m^3$. Con 58 mil filas ya no termina en tiempo razonable; con 3 millones, ni pensarlo.

## 7.3 · Sus opciones: SVM en su problema

Dos caminos, y conviene probar ambos:
- **SVM lineal** (`LinearSVC` / `LinearSVR`): escala a todo train. Su apuesta es la misma frontera plana que la logística, con otra pérdida.
- **SVM con kernel RBF** (`SVC` / `SVR`): en una **muestra** de train (8 000 filas, estratificada en clasificación). Hay que decirlo en el reporte. En la sección 11 hay una opción para entrenarla con mucho más datos en GPU.

Las SVM **necesitan variables escaladas** (`escalado=True`): la distancia y el margen dependen de las unidades.

In [ ]:
N_MUESTRA_SVM = 8_000            # ✋ EQUIPO: más filas = mejor SVM y más tiempo (crece más que lineal)
i_svm = muestra_train(N_MUESTRA_SVM)

if ES_REGRESION:
    for C in [0.01, 1]:
        evalua("SVM lineal", f"C={C}", LinearSVR(C=C, epsilon=0, dual=True, max_iter=5000, random_state=0), escalado=True)
    for C in [1, 10]:
        evalua("SVM RBF", f"C={C}, γ=scale, n={N_MUESTRA_SVM}", SVR(kernel="rbf", C=C, gamma="scale", epsilon=0.5),
               escalado=True, filas=i_svm)
else:
    for C in [0.01, 0.1]:
        evalua("SVM lineal", f"C={C}", LinearSVC(C=C, max_iter=5000, random_state=0), escalado=True)
    for C in [1, 10]:
        evalua("SVM RBF", f"C={C}, γ=scale, n={N_MUESTRA_SVM}", SVC(kernel="rbf", C=C, gamma="scale"),
               escalado=True, filas=i_svm)

Una SVM no entrega probabilidades sino una **distancia a la frontera** (`decision_function`). Para la AP y para el umbral basta, porque solo importa el orden; lo único que cambia es que el umbral ya no está entre 0 y 1.

> **Para su reporte (C3, C5).** Que la SVM con kernel usó una muestra y de qué tamaño; si perdió, ¿fue por su apuesta (frontera suave) o por haber visto menos datos? Comparen la SVM lineal con la logística: misma frontera, distinta pérdida.

---
# 8 · La comparativa: ¿quién gana y la diferencia es real?

## 8.1 · La tabla

Esta es la tabla que va al reporte: el mejor de cada familia y los baselines. La columna **brecha** es train menos validación (en regresión, al revés, para que siempre "brecha grande" signifique "sobreajusta").

In [ ]:
familias_modelo = [f for f in elegidos if not f.startswith("baseline")]
if len(familias_modelo) < 3:
    print(f"⚠ Llevan {len(familias_modelo)} familia(s) de modelos: la rúbrica pide al menos 3.")

filas = []
for f, info in elegidos.items():
    r = next(r for r in registro if r["familia"] == f and r["hiperparámetros"] == info["hiper"])
    brecha = r[f"{NOMBRE_METRICA} train"] - r[f"{NOMBRE_METRICA} val"]
    filas.append({**r, "brecha": brecha if MAYOR_ES_MEJOR else -brecha})
comparativa = pd.DataFrame(filas).set_index("familia").sort_values(f"{NOMBRE_METRICA} val", ascending=not MAYOR_ES_MEJOR)
comparativa.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.5 * len(comparativa) + 1))
colores = [COLORES["b"] if f.startswith("baseline") else COLORES["w"] for f in comparativa.index]
ax.barh(comparativa.index[::-1], comparativa[f"{NOMBRE_METRICA} val"][::-1], color=colores[::-1])
ax.set(xlabel=f"{NOMBRE_METRICA} en validación" + (" (más alto es mejor)" if MAYOR_ES_MEJOR else " (más bajo es mejor)"),
       title="El mejor de cada familia contra los baselines")
plt.show()

CANDIDATO = next(f for f in comparativa.index if not f.startswith("baseline"))
MEJOR_BASELINE = next(f for f in comparativa.index if f.startswith("baseline"))
RIVAL = next(f for f in comparativa.index if f != CANDIDATO)
print(f"candidato: {CANDIDATO} ({elegidos[CANDIDATO]['hiper']}) · rival más cercano: {RIVAL} · mejor baseline: {MEJOR_BASELINE}")

## 8.2 · Ejemplo: ¿la diferencia es más grande que el ruido?

Agosto es un solo mes. Si el candidato gana por 0.005, ¿ganaría también en otro mes? Una forma honesta de saberlo es el **bootstrap por bloques**: remuestrear **semanas completas** de validación (no filas sueltas, porque las horas de una misma semana se parecen entre sí) y recalcular la diferencia muchas veces.

In [ ]:
def bootstrap_por_bloques(y, s_a, s_b, bloques, replicas=300, semilla=0):
    # Diferencia de métrica (positiva = gana a) remuestreando bloques completos con reemplazo
    rng = np.random.default_rng(semilla)
    grupos = [np.flatnonzero(bloques == k) for k in np.unique(bloques)]
    signo = 1 if MAYOR_ES_MEJOR else -1
    difs = []
    for _ in range(replicas):
        idx = np.concatenate([grupos[k] for k in rng.integers(0, len(grupos), len(grupos))])
        difs.append(signo * (metrica(y[idx], s_a[idx]) - metrica(y[idx], s_b[idx])))
    return np.array(difs)


semana_va = ((va["hour"] - va["hour"].min()).dt.days // 7).to_numpy()
for otro in dict.fromkeys([RIVAL, MEJOR_BASELINE]):
    d = bootstrap_por_bloques(y_va, elegidos[CANDIDATO]["s_va"], elegidos[otro]["s_va"], semana_va)
    print(f"{CANDIDATO} contra {otro}: ventaja de {np.mean(d):+.3f} · intervalo 90 % [{np.quantile(d, 0.05):+.3f}, "
          f"{np.quantile(d, 0.95):+.3f}] · gana en {np.mean(d > 0):.0%} de las réplicas")

Si el intervalo contra el rival cruza el cero, **no tienen un ganador claro**: pueden elegir el más simple, el más rápido o el más fácil de explicar a la operación, y decirlo así en el reporte. Eso también es una conclusión válida.

## 8.3 · Ejemplo: ¿en qué se fija el ganador?

La **importancia por permutación** revuelve una variable a la vez en validación y mide cuánto empeora el modelo. Si empeora mucho, el modelo depende de esa variable. Sirve para la lectura con la teoría: ¿el modelo usa lo que esperarían?

In [ ]:
from sklearn.inspection import permutation_importance

info = elegidos[CANDIDATO]
A_va_c = Z_va if info["escalado"] else X_va
i_pi = np.random.default_rng(2).choice(len(y_va), min(5_000, len(y_va)), replace=False)


class _Puntua:
    # Envuelve al modelo para que permutation_importance use la métrica de este notebook
    def __init__(self, m): self.m = m
    def fit(self, *a): return self

def _calif(est, A, y):
    s = metrica(y, puntaje(est.m, A))
    return s if MAYOR_ES_MEJOR else -s

imp = permutation_importance(_Puntua(info["modelo"]), _filas(A_va_c, i_pi), y_va[i_pi], scoring=_calif,
                             n_repeats=3, random_state=0)
importancias = pd.Series(imp.importances_mean, index=X_va.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 0.3 * len(importancias) + 1))
ax.barh(importancias.index, importancias.values, color=COLORES["w"])
ax.set(xlabel=f"cuánto empeora {NOMBRE_METRICA} al revolver la variable", title=f"Importancia por permutación · {CANDIDATO}")
plt.show()

> **Para su reporte (C3, C5).** La tabla comparativa (familia, hiperparámetros elegidos, métrica en train y en val); el diagnóstico de sesgo y varianza con la columna **brecha**; si la diferencia con el segundo es real (intervalo del bootstrap); y la frase de sesgo inductivo de la sección 3, ahora confirmada o corregida con sus números.

---
# 9 · Métrica y umbral para operar

La AP sirvió para elegir modelo. Para operar hace falta una decisión concreta: **a partir de qué puntaje se manda la camioneta**. Eso no lo decide la estadística sino la operación. Tres formas razonables de fijarlo, todas **en validación**:

| `CRITERIO_UMBRAL` | Pregunta que contesta | Qué tienen que fijar |
|---|---|---|
| `"presupuesto"` | "Podemos atender tantas estaciones por hora, ¿cuáles?" | `ALERTAS_POR_HORA` |
| `"costos"` | "Un evento sin atender cuesta lo que 5 viajes en vano, ¿dónde está el equilibrio?" | `COSTO_FP`, `COSTO_FN` |
| `"recall"` | "Queremos atrapar al menos el 60 % de los eventos, ¿con cuántas alertas?" | `RECALL_MIN` |

Los números de abajo son **supuestos de ejemplo**, no datos de MiBici. Cámbienlos por los que puedan defender y expliquen de dónde salen.

*Si su problema es A (regresión), no hay umbral: la celda de abajo les muestra en qué horas se equivoca más el modelo, que es lo que necesitan para justificar MAE o RMSE.*

In [ ]:
CRITERIO_UMBRAL = "presupuesto"   # ✋ EQUIPO: "presupuesto", "costos" o "recall"
ALERTAS_POR_HORA = 6 if not EN_PANEL else 20     # ✋ supuesto: estaciones que la operación puede atender por hora
COSTO_FP, COSTO_FN = 1, 5                        # ✋ supuesto: costo relativo de un viaje en vano y de un evento sin atender
RECALL_MIN = 0.60                                # ✋ supuesto: fracción mínima de eventos que hay que atrapar

s_va_c = elegidos[CANDIDATO]["s_va"]

if ES_REGRESION:
    por_hora = pd.DataFrame({"hora": va["hour_of_day"].to_numpy(),
                             "modelo": np.abs(y_va - s_va_c),
                             "baseline": np.abs(y_va - elegidos[MEJOR_BASELINE]["s_va"])}).groupby("hora").mean()
    ax = por_hora.plot(marker="o", figsize=(9, 4), title="Error absoluto medio por hora del día (val)")
    ax.set(ylabel="bicis/h")
    plt.show()
    print(f"{CANDIDATO}: MAE val = {mean_absolute_error(y_va, s_va_c):.3f} · RMSE val = {np.sqrt(mean_squared_error(y_va, s_va_c)):.3f}")
    print(f"sesgo medio (predicción − real) = {np.mean(s_va_c - y_va):+.3f} bicis/h")
    UMBRAL_ELEGIDO = None
else:
    cand = np.unique(np.quantile(s_va_c, np.linspace(0, 1, 801)))
    def resumen_umbral(t, s=s_va_c, y=y_va):
        a = s >= t
        return {"umbral": t, "% alertas": a.mean(), "alertas por hora": a.mean() * N_ESTACIONES,
                "precision": precision_score(y, a, zero_division=0), "recall": recall_score(y, a),
                "costo": COSTO_FP * np.sum(a & (y == 0)) + COSTO_FN * np.sum(~a & (y == 1))}
    tabla_u = pd.DataFrame([resumen_umbral(t) for t in cand])
    opciones = {
        "presupuesto": tabla_u[tabla_u["alertas por hora"] <= ALERTAS_POR_HORA]["umbral"].min(),
        "costos": tabla_u.loc[tabla_u["costo"].idxmin(), "umbral"],
        "recall": tabla_u[tabla_u["recall"] >= RECALL_MIN]["umbral"].max(),
    }
    UMBRAL_ELEGIDO = float(opciones[CRITERIO_UMBRAL])
    print(pd.DataFrame({k: resumen_umbral(v) for k, v in opciones.items()}).T.round(3))

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(tabla_u["% alertas"], tabla_u["precision"], label="precision", color=COLORES["w"])
    ax.plot(tabla_u["% alertas"], tabla_u["recall"], label="recall", color=COLORES["error"])
    colores_op = {"presupuesto": COLORES["lambda"], "costos": COLORES["yhat"], "recall": COLORES["b"]}
    for k, v in opciones.items():
        ax.axvline((s_va_c >= v).mean(), ls="--" if k == CRITERIO_UMBRAL else ":", color=colores_op[k], label=f"criterio: {k}")
    ax.set(xlabel="% de estaciones-hora con alerta", title=f"{CANDIDATO}: qué se gana y qué se pierde al alertar más (val)")
    ax.legend()
    plt.show()
    print(f"\nUMBRAL_ELEGIDO ({CRITERIO_UMBRAL}) = {UMBRAL_ELEGIDO:.4f}")

**Otra forma de gastar el presupuesto.** Un umbral fijo da `ALERTAS_POR_HORA` *en promedio*, pero en una hora pico puede pedir muchas más y en la madrugada ninguna. La alternativa es atender, cada hora, las `ALERTAS_POR_HORA` estaciones con puntaje más alto. Comparen las dos políticas:

In [ ]:
if not ES_REGRESION:
    def top_k_por_hora(d, s, k):
        rango = pd.Series(s, index=d.index).groupby(d["hour"].to_numpy()).rank(ascending=False, method="first")
        return (rango <= k).to_numpy()

    a_umbral = s_va_c >= UMBRAL_ELEGIDO
    a_topk = top_k_por_hora(va, s_va_c, ALERTAS_POR_HORA)
    for nombre, a in [("umbral fijo", a_umbral), (f"top {ALERTAS_POR_HORA} por hora", a_topk)]:
        por_h = pd.Series(a).groupby(va["hour"].to_numpy()).sum()
        print(f"{nombre:16s} · precision {precision_score(y_va, a):.3f} · recall {recall_score(y_va, a):.3f} · "
              f"alertas por hora: mediana {por_h.median():.0f}, máximo {por_h.max():.0f}")

El umbral fijo suele ganar en precision porque concentra las alertas en las horas donde de verdad hay eventos, pero a cambio pide en una sola hora más estaciones de las que una cuadrilla puede atender (miren el máximo). El "top por hora" nunca se pasa del presupuesto, pero gasta alertas de madrugada donde casi no pasa nada. ¿Cuál puede operar MiBici? Esa discusión vale un párrafo del reporte.

> **Para su reporte (C4).** La métrica con la que eligieron modelo y por qué; el criterio de umbral, **con los supuestos numéricos** (cuántas estaciones por hora, o cuánto cuesta cada error) y de dónde salen; precision y recall resultantes en validación. En regresión: MAE o RMSE por el costo del error y en qué horas falla más el modelo.

---
# 10 · Test, una sola vez

Hasta aquí todo se decidió con validación: problema, variables, modelo, hiperparámetros, métrica y umbral. Ahora, y solo ahora, se mide en test. Cambien `REPORTE_FINAL` a `True` y ejecuten la celda **una vez**.

**Opción: reentrenar con train + val.** Una práctica común es volver a ajustar el modelo elegido (con los mismos hiperparámetros) usando train y validación juntos, para que vea los datos más recientes. Es válido siempre que no cambie ninguna decisión. Si lo hacen, díganlo; la celda trae la bandera `REENTRENAR_CON_VAL`. El umbral se queda como se fijó en validación.

Si después de ver test cambian algo, ya no es una evaluación honesta: díganlo en el reporte.

In [ ]:
REPORTE_FINAL = False          # ✋ EQUIPO: True solo cuando todo esté decidido
REENTRENAR_CON_VAL = False     # ✋ EQUIPO: opcional (ver arriba)

resultado_test = None
if REPORTE_FINAL:
    info = elegidos[CANDIDATO]
    modelo = info["modelo"]
    A_te = Z_te if info["escalado"] else X_te
    if REENTRENAR_CON_VAL and "RBF" not in CANDIDATO and "GPU" not in CANDIDATO and "red" not in CANDIDATO:
        from sklearn.base import clone
        A_trva = np.vstack([Z_tr, Z_va]) if info["escalado"] else pd.concat([X_tr, X_va])
        nuevo = clone(modelo)
        if hasattr(modelo, "best_iteration"):          # XGBoost: conserva los árboles que eligió validación
            nuevo.set_params(n_estimators=modelo.best_iteration + 1, early_stopping_rounds=None)
        modelo = nuevo.fit(A_trva, np.concatenate([y_tr, y_va]))
        print("modelo reentrenado con train + val\n")
    elif REENTRENAR_CON_VAL:
        print("Este candidato se entrenó con una muestra o en GPU: se evalúa tal cual, sin reentrenar.\n")
    s_te = puntaje(modelo, A_te)
    s_te_base = elegidos[MEJOR_BASELINE]["regla"](te)
    resultado_test = {"modelo": CANDIDATO, "hiperparámetros": info["hiper"],
                      f"{NOMBRE_METRICA} test": metrica(y_te, s_te),
                      f"{NOMBRE_METRICA} test · {MEJOR_BASELINE}": metrica(y_te, s_te_base)}
    if not ES_REGRESION:
        a = s_te >= UMBRAL_ELEGIDO
        resultado_test.update({"% alertas": a.mean(), "alertas por hora": a.mean() * N_ESTACIONES,
                               "precision": precision_score(y_te, a, zero_division=0), "recall": recall_score(y_te, a)})
        print(pd.DataFrame(confusion_matrix(y_te, a), index=["real: sin evento", "real: evento"],
                           columns=["predice: no", "predice: alerta"]), "\n")
    else:
        resultado_test["RMSE test" if METRICA_REG == "MAE" else "MAE test"] = (
            float(np.sqrt(mean_squared_error(y_te, s_te))) if METRICA_REG == "MAE" else mean_absolute_error(y_te, s_te))
    print(pd.Series(resultado_test).to_string())
else:
    print("Test sin tocar. Cambien REPORTE_FINAL a True cuando hayan decidido todo con validación.")

Es normal que test salga un poco distinto de validación: es otro mes, con otra prevalencia y otro clima. Si sale **muy** distinto, esa es una limitación que vale la pena discutir (sección 12).

---
# 11 · *Opcional:* modelos en GPU

Esta sección es **opcional** y no se califica por sí misma. Sirve para quitar la limitación más grande del notebook: entrenar con una **muestra** de los datos porque la CPU no alcanza. Con GPU pueden entrenar boosting sobre los 3 millones de filas del panel, o una SVM con kernel con decenas de miles de filas en lugar de 8 000.

**Cómo activar la GPU en Colab:** *Entorno de ejecución*, opción *Cambiar tipo de entorno de ejecución*, elijan **T4 GPU** y guarden. Eso **reinicia** el entorno: vuelvan a ejecutar todo desde arriba (*Entorno de ejecución*, opción *Ejecutar todo*). La GPU gratuita de Colab tiene horario limitado; si no está disponible, prueben más tarde.

Tienen tres opciones, cada una con su bandera. Si la activan y no hay GPU, las dos primeras corren en CPU (más lento) y la tercera avisa.

| Opción | Qué hace | Cuenta como familia del módulo |
|---|---|---|
| **G1 · XGBoost** | boosting de árboles sobre **todo** train (en C y D, 3 millones de filas sin submuestrear) | sí: es boosting |
| **G2 · SVM RBF con cuML** | la misma SVM con kernel, con 10 veces más filas | sí: es SVM |
| **G3 · Red neuronal (PyTorch)** | un perceptrón de dos capas | **no**: es un modelo de otro módulo; úsenlo solo como comparación extra |

In [ ]:
import shutil, subprocess

HAY_GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
if HAY_GPU:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("No hay GPU en este entorno. Las opciones G1 y G3 pueden correr en CPU (más lento); G2 necesita GPU.")

GPU_XGBOOST = False     # ✋ G1
GPU_SVM = False         # ✋ G2
GPU_RED = False         # ✋ G3

## 11.1 · G1 · XGBoost sobre todo train

XGBoost es otra implementación de boosting de árboles, muy usada en la industria, que puede entrenar en GPU con `device="cuda"`. Usamos **todo** train (en C y D, sin la submuestra de 300 mil filas) y fijamos el número de árboles con **parada temprana en validación**: se detiene cuando 50 árboles seguidos no mejoran la métrica de validación. Es una forma legítima de elegir un hiperparámetro con validación.

In [ ]:
if GPU_XGBOOST:
    try:
        import xgboost as xgb
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
        import xgboost as xgb

    _, X_tr_todo, y_tr_todo = prepara_train_completo()            # sin submuestra
    print(f"XGBoost con {len(X_tr_todo):,} filas de train en {'GPU' if HAY_GPU else 'CPU'}")
    comunes = dict(n_estimators=2000, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                   tree_method="hist", device="cuda" if HAY_GPU else "cpu", early_stopping_rounds=50, random_state=0)
    if ES_REGRESION:
        xgb_m = xgb.XGBRegressor(objective="reg:absoluteerror" if METRICA_REG == "MAE" else "reg:squarederror",
                                 eval_metric="mae" if METRICA_REG == "MAE" else "rmse", **comunes)
    else:
        xgb_m = xgb.XGBClassifier(eval_metric="aucpr", **comunes)
    evalua("boosting (XGBoost, todo train)", "lr=0.05, prof=8, parada temprana", xgb_m, escalado=False,
           A=X_tr_todo, y=y_tr_todo, eval_set=[(X_va, y_va)], verbose=False)
    print(f"árboles elegidos por validación: {xgb_m.best_iteration + 1}")
else:
    print("G1 desactivada (GPU_XGBOOST = False).")

## 11.2 · G2 · SVM con kernel en GPU (cuML)

cuML es la versión en GPU de buena parte de `scikit-learn` y viene instalada en los entornos con GPU de Colab. Con ella, la SVM RBF que en CPU solo aguantaba 8 000 filas puede entrenar con 80 000 en un tiempo parecido. Así pueden separar las dos explicaciones de la sección 7: ¿la SVM perdía por su apuesta o por haber visto pocos datos?

In [ ]:
N_MUESTRA_SVM_GPU = 80_000     # ✋ EQUIPO

if GPU_SVM:
    if not HAY_GPU:
        print("G2 necesita GPU: activen T4 GPU en Colab y vuelvan a ejecutar todo.")
    else:
        try:
            import cuml
        except ImportError:
            cuml = None
            print("cuML no está instalado. En Colab con GPU suele venir; si no: !pip install cuml-cu12 (tarda varios minutos).")
        if cuml is not None:
            from cuml.svm import SVC as SVC_GPU, SVR as SVR_GPU

            class _SVMGPU:
                # Misma interfaz que sklearn, con entrada float32 (lo que cuML espera)
                def __init__(self, base): self.base = base
                def fit(self, A, y): self.base.fit(np.asarray(A, np.float32), np.asarray(y, np.float32)); return self
                def predict(self, A): return np.asarray(self.base.predict(np.asarray(A, np.float32)))
                def decision_function(self, A): return np.asarray(self.base.decision_function(np.asarray(A, np.float32)))

            i_gpu = muestra_train(N_MUESTRA_SVM_GPU)
            for C in [1, 10]:
                base = (SVR_GPU(kernel="rbf", C=C, gamma="scale", epsilon=0.5) if ES_REGRESION
                        else SVC_GPU(kernel="rbf", C=C, gamma="scale"))
                evalua("SVM RBF (GPU)", f"C={C}, γ=scale, n={len(i_gpu)}", _SVMGPU(base), escalado=True, filas=i_gpu)
else:
    print("G2 desactivada (GPU_SVM = False).")

## 11.3 · G3 · Una red neuronal chica (PyTorch), como comparación extra

Un perceptrón multicapa: dos capas ocultas de 64 neuronas con ReLU. Para clasificación se entrena con la misma log-loss que la logística (`BCEWithLogitsLoss`); para regresión, con error absoluto o cuadrático según `METRICA_REG`. Una logística es una red **sin** capas ocultas, así que esta comparación responde una pregunta concreta: ¿las interacciones no lineales que la red puede aprender valen algo en este problema? Los árboles ya responden eso dentro del módulo, así que **no cuenta** como una de las tres familias.

In [ ]:
EPOCAS_RED = 4         # ✋ EQUIPO

if GPU_RED:
    try:
        import torch
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
        import torch
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

    class RedChica:
        # Interfaz tipo sklearn para que evalua() la trate como a cualquier otro modelo
        def __init__(self, ocultas=64, epocas=EPOCAS_RED, lote=2048, lr=1e-3, semilla=0):
            self.ocultas, self.epocas, self.lote, self.lr, self.semilla = ocultas, epocas, lote, lr, semilla

        def fit(self, A, y):
            torch.manual_seed(self.semilla)
            A = torch.tensor(np.asarray(A, np.float32), device=dispositivo)
            y = torch.tensor(np.asarray(y, np.float32), device=dispositivo)
            self.media = float(y.mean()) if ES_REGRESION else 0.0
            self.red = torch.nn.Sequential(torch.nn.Linear(A.shape[1], self.ocultas), torch.nn.ReLU(),
                                           torch.nn.Linear(self.ocultas, self.ocultas), torch.nn.ReLU(),
                                           torch.nn.Linear(self.ocultas, 1)).to(dispositivo)
            if ES_REGRESION:
                perdida = torch.nn.L1Loss() if METRICA_REG == "MAE" else torch.nn.MSELoss()
            else:
                perdida = torch.nn.BCEWithLogitsLoss()
            opt = torch.optim.Adam(self.red.parameters(), lr=self.lr)
            for epoca in range(self.epocas):
                orden = torch.randperm(len(y), device=dispositivo)
                total = 0.0
                for k in range(0, len(y), self.lote):
                    i = orden[k:k + self.lote]
                    opt.zero_grad()
                    l = perdida(self.red(A[i]).squeeze(1) + self.media, y[i])
                    l.backward()
                    opt.step()
                    total += l.item() * len(i)
                print(f"  época {epoca + 1}: pérdida de train = {total / len(y):.4f}")
            return self

        def _salida(self, A):
            with torch.no_grad():
                A = torch.tensor(np.asarray(A, np.float32), device=dispositivo)
                return np.concatenate([(self.red(A[k:k + 65536]).squeeze(1) + self.media).cpu().numpy()
                                       for k in range(0, len(A), 65536)])

        def predict(self, A):
            return self._salida(A)

        def predict_proba(self, A):
            p = 1 / (1 + np.exp(-self._salida(A)))
            return np.c_[1 - p, p]

    if EN_PANEL:
        _, X_red, y_red = prepara_train_completo()                # todo train
        A_red = escalador.transform(X_red)
    else:
        A_red, y_red = Z_tr, y_tr
    print(f"red neuronal en {dispositivo} con {len(y_red):,} filas")
    evalua("red neuronal (extra)", f"2×64, {EPOCAS_RED} épocas", RedChica(), escalado=True, A=A_red, y=y_red)
else:
    print("G3 desactivada (GPU_RED = False).")

Si activaron alguna opción, **vuelvan a ejecutar la sección 8** para que la comparativa y el umbral las incluyan. Si un modelo de GPU queda como candidato, la sección 10 lo usa sin cambios.

> **Para su reporte.** Si usaron GPU: qué modelo, con cuántas filas, cuánto tardó y si cambió el ganador. Si la red neuronal ganó, díganlo, pero su recomendación tiene que salir de las familias del módulo (o explicar por qué recomiendan la red de todos modos).

---
# 12 · Limitaciones y riesgos

## 12.1 · Ejemplo: ¿el modelo funciona igual en todas las estaciones?

Un número promedio esconde diferencias. Partimos las estaciones en cuatro grupos según su demanda en train y medimos el candidato en cada grupo.

In [ ]:
demanda = tr.groupby("station_id")["departures"].mean()
grupo_est = pd.qcut(demanda, 4, labels=["1 · poca demanda", "2", "3", "4 · mucha demanda"])
g_va = va["station_id"].map(grupo_est).to_numpy()
s_c, s_b = elegidos[CANDIDATO]["s_va"], elegidos[MEJOR_BASELINE]["s_va"]


def metrica_grupo(y, s):
    # En clasificación, un grupo sin eventos no tiene AP definida
    return np.nan if (not ES_REGRESION and y.sum() == 0) else metrica(y, s)


filas = []
for g in grupo_est.cat.categories:
    m = g_va == g
    fila = {"grupo": g, "estaciones": int((grupo_est == g).sum()), f"{NOMBRE_METRICA} candidato": metrica_grupo(y_va[m], s_c[m]),
            f"{NOMBRE_METRICA} baseline": metrica_grupo(y_va[m], s_b[m])}
    if not ES_REGRESION:
        a = s_c[m] >= UMBRAL_ELEGIDO
        fila.update({"prevalencia": y_va[m].mean(), "% del total de alertas": a.sum() / max(1, (s_c >= UMBRAL_ELEGIDO).sum()),
                     "recall": recall_score(y_va[m], a, zero_division=0)})
    filas.append(fila)
if EN_PANEL:
    for mun in sorted(va["municipality"].unique()):
        m = (va["municipality"] == mun).to_numpy()
        filas.append({"grupo": f"municipio {mun}", "estaciones": int(va.loc[m, "station_id"].nunique()),
                      f"{NOMBRE_METRICA} candidato": metrica_grupo(y_va[m], s_c[m]), f"{NOMBRE_METRICA} baseline": metrica_grupo(y_va[m], s_b[m]),
                      "prevalencia": np.nan if ES_REGRESION else y_va[m].mean()})
pd.DataFrame(filas).set_index("grupo").round(3)

Si las alertas se concentran en las estaciones de mucha demanda, las de poca demanda casi nunca reciben camioneta, aunque también se vacíen. Eso puede ser razonable (atienden a más gente) o injusto (suelen estar en zonas con menos alternativas de transporte). Es una decisión de la operación, pero el modelo la está tomando sin que nadie la haya discutido. Ese es el tipo de riesgo que pide la rúbrica.

## 12.2 · Sus opciones: limitaciones y riesgos para elegir

Elijan **al menos dos limitaciones y un riesgo**, los que apliquen a su problema, y apóyenlos con un número de su notebook cuando se pueda.

**Limitaciones (de los datos o del modelo)**
- **La etiqueta es un proxy.** `future_net` mide salidas menos llegadas, no si la estación se quedó vacía. Sin el inventario de bicis no sabemos si el desbalance de verdad dejó a alguien sin servicio.
- **Demanda censurada.** Cuando la estación está vacía nadie puede sacar bici, así que los datos registran *menos* salidas de las que hubo en demanda. El modelo aprende la demanda atendida, no la real.
- **El rebalanceo ya ocurrió en los datos.** Si la operación movió bicis en junio, el flujo observado ya trae esa intervención mezclada.
- **Pocos meses y cambio de temporada.** Lluvias de verano, vacaciones y regreso a clases cambian el patrón: test (septiembre o ago–sep) no se parece del todo a train. Comparen la prevalencia entre particiones (sección 2.2).
- **Variables que no están:** clima, eventos masivos, días festivos, obras.
- **Muestra para entrenar.** La SVM con kernel (y, en C y D, todos los modelos) vio una muestra de train.

**Riesgos (éticos, operativos o de datos) con mitigación**
- **Desigualdad en el servicio** (tabla de 12.1): reservar parte del presupuesto de alertas para estaciones de poca demanda y vigilar el recall por grupo cada mes.
- **Ciclo de retroalimentación:** si el modelo manda camionetas y estas evitan los eventos, los datos futuros tendrán menos eventos donde el modelo acertó, y el modelo reentrenado "aprenderá" que ahí no pasa nada. Mitigación: registrar las intervenciones y usarlas como variable, o dejar un grupo de control.
- **Deriva:** una estación nueva, una ciclovía o una estación que cambia de capacidad. Mitigación: monitorear la métrica semanal contra el baseline y reentrenar cuando el modelo deje de ganarle.
- **Exceso de confianza:** si la operación confía ciegamente en el modelo, deja de usar su conocimiento local. Mitigación: presentar el puntaje como prioridad, no como orden.

> **Para su reporte (C6).** Dos limitaciones y un riesgo, cada uno en una o dos frases, con su mitigación y, si se puede, un número que lo respalde.

---
# 13 · Del notebook al reporte

## 13.1 · Los números de su reporte

Esta celda junta todo lo que el reporte necesita. Cópienlos de aquí: los números del reporte tienen que salir de su notebook.

In [ ]:
print(f"PROBLEMA {PROBLEMA}: {cfg['nombre']}")
print(f"Datos: {'panel completo' if EN_PANEL else 'recorte'} · {N_ESTACIONES} estaciones · variables '{CONJUNTO}' ({X_tr.shape[1]})")
print(f"Partición: train {tr['hour'].min():%d-%m-%Y}–{tr['hour'].max():%d-%m-%Y} ({len(X_tr):,} filas"
      f"{', submuestra' if EN_PANEL else ''}) · val {va['hour'].min():%d-%m-%Y}–{va['hour'].max():%d-%m-%Y} · "
      f"test {te['hour'].min():%d-%m-%Y}–{te['hour'].max():%d-%m-%Y}")
if not ES_REGRESION:
    print(f"Prevalencia train / val / test: {y_tr.mean():.2%} / {y_va.mean():.2%} / {y_te.mean():.2%}")
print(f"Métrica para elegir modelo: {NOMBRE_METRICA}")
print(f"\nCandidato: {CANDIDATO} ({elegidos[CANDIDATO]['hiper']}) · {NOMBRE_METRICA} val = {elegidos[CANDIDATO]['val']:.3f}")
print(f"Mejor baseline: {MEJOR_BASELINE} · {NOMBRE_METRICA} val = {elegidos[MEJOR_BASELINE]['val']:.3f}")
if not ES_REGRESION:
    print(f"Umbral ({CRITERIO_UMBRAL}): {UMBRAL_ELEGIDO:.4f}")
print("\nTest:", "sin ejecutar (REPORTE_FINAL = False)" if resultado_test is None else "")
if resultado_test is not None:
    print(pd.Series(resultado_test).to_string())
print()
comparativa.round(3)

## 13.2 · Esqueleto del reporte (2 páginas)

Un orden que cabe en dos páginas carta a 11 pt:

1. **Problema (C1)** · 3–4 líneas: la decisión, la etiqueta, la unidad de predicción, los datos y sus límites.
2. **Cómo evaluamos (C2)** · 4–5 líneas: partición temporal con fechas, qué se ajustó solo con train, cómo revisaron fugas, el baseline con su número.
3. **Comparativa (C3)** · la tabla de la sección 8 (familia, hiperparámetros, train, val, brecha) y 2–3 líneas.
4. **Métrica y umbral (C4)** · por qué esta métrica, el criterio de umbral con sus supuestos, y el resultado en test (una sola vez).
5. **Lectura (C5)** · sesgo y varianza con la brecha y la curva del árbol; el sesgo inductivo de cada modelo y por qué ganó el que ganó; si la diferencia es real.
6. **Limitaciones y riesgos (C6)** · dos limitaciones y un riesgo con su mitigación.
7. **Recomendación** · en una frase: qué desplegarían, con qué umbral y qué vigilarían.
8. **Quién hizo qué** · una línea.

**Una figura que vale la pena incluir:** la curva de profundidad del árbol (sesgo y varianza en una imagen) o la de precision y recall contra el % de alertas (el umbral en una imagen). Con una basta.

## 13.3 · Lista de verificación

**Notebook**
- [ ] Corre completo con *Reiniciar y ejecutar todo*.
- [ ] Todo lo aprendido de los datos (umbral del evento, escalador) se ajusta solo con train.
- [ ] Ninguna variable de la hora *t* ni del futuro entre las entradas (tabla de la sección 2.4).
- [ ] Al menos 3 familias de modelos del módulo, cada una con su rejilla en validación.
- [ ] El umbral (o la métrica de regresión) se fijó en validación.
- [ ] Test se ejecutó una vez, al final, con `REPORTE_FINAL = True`.

**Reporte**
- [ ] Pregunta, etiqueta, unidad de predicción y límites de los datos.
- [ ] Partición, cómo evitaron fugas y el baseline con su número.
- [ ] Tabla comparativa con hiperparámetros elegidos.
- [ ] Métrica y umbral justificados con costos o presupuesto, y el resultado en test.
- [ ] Sesgo y varianza **y** sesgo inductivo, con números.
- [ ] Dos limitaciones y un riesgo con mitigación.
- [ ] Nombres y quién hizo qué.